In [2]:
from pathlib import Path
from urllib.parse import urlparse
import json
import boto3
import pandas as pd

PROJECT_ROOT = Path.cwd()
config_path = PROJECT_ROOT / "project_config.json"
project_config = json.loads(config_path.read_text(encoding="utf-8"))

AWS_REGION = project_config["aws_region"]
PROJECT_BUCKET = project_config["s3_bucket"]

aws_session = boto3.Session(region_name=AWS_REGION)
s3_client = aws_session.client("s3")
sagemaker_client = aws_session.client("sagemaker")

# Load production inputs only; their labels remain separate.
production_info = project_config["s3_data"]["production"]
location = urlparse(production_info["uri"])

response = s3_client.get_object(
    Bucket=location.netloc,
    Key=location.path.lstrip("/"),
    VersionId=production_info["version_id"]
)
try:
    production_df = pd.read_csv(
        response["Body"],
        dtype={"ticket_id": "string"},
        low_memory=False
    )
finally:
    response["Body"].close()

assert len(production_df) == 17712
assert production_df["ticket_id"].notna().all()
assert production_df["ticket_id"].is_unique
assert "queue" not in production_df.columns

print("Production tickets:", len(production_df))
print("Columns:", production_df.columns.tolist())
print("Selected model:", project_config["final_model"]["training_job_name"])

Production tickets: 17712
Columns: ['ticket_id', 'subject', 'body', 'language', 'priority', 'type']
Selected model: aai540-group3-lr-20261004-003353


In [2]:
import hashlib
import shutil

# Prepare the folder for our prediction script.
INFERENCE_SOURCE_DIR = PROJECT_ROOT / "src" / "ticket_inference"
INFERENCE_SOURCE_DIR.mkdir(parents=True, exist_ok=True)

# Use the same package versions as training.
shutil.copyfile(
    PROJECT_ROOT / "src" / "ticket_training" / "requirements.txt",
    INFERENCE_SOURCE_DIR / "requirements.txt"
)

batch_dir = PROJECT_ROOT / "data" / "batch"
batch_dir.mkdir(parents=True, exist_ok=True)
batch_input_path = batch_dir / "production.jsonl"

# Include only the fields needed for prediction and monitoring.
input_columns = ["ticket_id", "subject", "body", "language"]
batch_records = (
    production_df[input_columns]
    .fillna("")
    .to_dict("records")
)

largest_record_bytes = 0

with batch_input_path.open("w", encoding="utf-8") as file:
    for record in batch_records:
        line = json.dumps(
            record, ensure_ascii=False, allow_nan=False
        ) + "\n"
        largest_record_bytes = max(
            largest_record_bytes, len(line.encode("utf-8"))
        )
        file.write(line)

# Each record must fit within our planned 1 MB request size.
assert largest_record_bytes < 1_000_000

model_job_name = project_config["final_model"]["training_job_name"]
batch_input_key = f"batch/{model_job_name}/input/production.jsonl"

s3_client.upload_file(
    str(batch_input_path), PROJECT_BUCKET, batch_input_key
)

metadata = s3_client.head_object(
    Bucket=PROJECT_BUCKET,
    Key=batch_input_key
)
assert metadata["ContentLength"] == batch_input_path.stat().st_size
assert metadata.get("VersionId") not in (None, "null")

project_config.setdefault("deployment", {}).update({
    "mode": "batch_transform",
    "input": {
        "uri": f"s3://{PROJECT_BUCKET}/{batch_input_key}",
        "version_id": metadata["VersionId"],
        "sha256": hashlib.sha256(
            batch_input_path.read_bytes()
        ).hexdigest(),
        "rows": len(batch_records)
    }
})

config_path.write_text(
    json.dumps(project_config, indent=2),
    encoding="utf-8"
)

print("Production records prepared:", len(batch_records))
print("Largest record:", largest_record_bytes, "bytes")
print("Batch input uploaded:", project_config["deployment"]["input"]["uri"])
print("Inference script folder:", INFERENCE_SOURCE_DIR)

Production records prepared: 17712
Largest record: 2994 bytes
Batch input uploaded: s3://aai540-group3-tickets-390568313988-us-east-1/batch/aai540-group3-lr-20261004-003353/input/production.jsonl
Inference script folder: /home/sagemaker-user/AAI-540-Project/multilingual-it-ticket-routing-mlops/src/ticket_inference


In [1]:
%%writefile src/ticket_inference/inference.py
import hashlib
import json
from pathlib import Path

import joblib


CONTENT_TYPE = "application/jsonlines"


def model_fn(model_dir):
    """Load the trained pipeline once when the prediction server starts."""
    model_path = Path(model_dir) / "model.joblib"

    return {
        "pipeline": joblib.load(model_path),
        "model_sha256": hashlib.sha256(
            model_path.read_bytes()
        ).hexdigest(),
    }


def input_fn(request_body, content_type):
    """Read one or more JSON Lines ticket records."""
    media_type = content_type.split(";", 1)[0].strip().lower()
    if media_type != CONTENT_TYPE:
        raise ValueError(f"Expected {CONTENT_TYPE}")

    if isinstance(request_body, bytes):
        request_body = request_body.decode("utf-8")

    records = [
        json.loads(line)
        for line in request_body.splitlines()
        if line.strip()
    ]

    if not records:
        raise ValueError("No ticket records provided")

    required_fields = {"ticket_id", "subject", "body", "language"}

    for record in records:
        if not isinstance(record, dict):
            raise ValueError("Each ticket must be a JSON object")

        if not required_fields.issubset(record):
            raise ValueError("A ticket is missing required fields")

        if "queue" in record:
            raise ValueError("Prediction inputs must not contain queue labels")

        if not isinstance(record["ticket_id"], str):
            raise ValueError("ticket_id must be a string")
        if not record["ticket_id"].strip():
            raise ValueError("ticket_id must not be empty")

        for field in ["subject", "body", "language"]:
            value = record[field]
            if value is not None and not isinstance(value, str):
                raise ValueError(f"{field} must be a string or null")

    return records


def predict_fn(records, model):
    """Apply the same text preparation used during training."""
    subjects = [(record["subject"] or "").strip() for record in records]
    bodies = [(record["body"] or "").strip() for record in records]

    texts = [
        (subject + " " + body).strip()
        for subject, body in zip(subjects, bodies)
    ]

    if any(not text for text in texts):
        raise ValueError("Each ticket needs a nonempty subject or body")

    pipeline = model["pipeline"]
    probabilities = pipeline.predict_proba(texts)
    classes = pipeline.classes_
    winning_indices = probabilities.argmax(axis=1)

    results = []

    for index, record in enumerate(records):
        winning_index = int(winning_indices[index])
        subject = subjects[index]
        body = bodies[index]
        text = texts[index]

        results.append({
            "ticket_id": record["ticket_id"],
            "language": (
                (record["language"] or "").strip().lower() or "unknown"
            ),
            "predicted_queue": str(classes[winning_index]),
            "max_probability": float(
                probabilities[index, winning_index]
            ),
            "has_subject": int(bool(subject)),
            "subject_char_count": len(subject),
            "body_char_count": len(body),
            "text_char_count": len(text),
            "subject_word_count": len(subject.split()),
            "body_word_count": len(body.split()),
            "text_word_count": len(text.split()),
            "model_sha256": model["model_sha256"],
        })

    return results


def output_fn(predictions, accept):
    """Return one JSON line per ticket, preserving input order."""
    media_type = accept.split(";", 1)[0].strip().lower()
    if media_type != CONTENT_TYPE:
        raise ValueError(f"Expected response type {CONTENT_TYPE}")

    response_body = "\n".join(
        json.dumps(record, ensure_ascii=False, allow_nan=False)
        for record in predictions
    )

    return response_body, CONTENT_TYPE

Writing src/ticket_inference/inference.py


In [3]:
import hashlib
import importlib.util
import tarfile
from io import BytesIO

# Download the selected model version.
final_model_info = project_config["final_model"]
model_location = urlparse(final_model_info["model_uri"])

response = s3_client.get_object(
    Bucket=model_location.netloc,
    Key=model_location.path.lstrip("/"),
    VersionId=final_model_info["model_version_id"]
)
try:
    model_archive_bytes = response["Body"].read()
finally:
    response["Body"].close()

assert hashlib.sha256(model_archive_bytes).hexdigest() == (
    final_model_info["model_archive_sha256"]
)

# Read only model.joblib from the archive.
local_model_dir = PROJECT_ROOT / "data" / "deployment" / "model"
local_model_dir.mkdir(parents=True, exist_ok=True)

with tarfile.open(
    fileobj=BytesIO(model_archive_bytes), mode="r:gz"
) as archive:
    matches = [
        member for member in archive.getmembers()
        if member.isfile() and Path(member.name).name == "model.joblib"
    ]
    assert len(matches) == 1

    with archive.extractfile(matches[0]) as model_file:
        (local_model_dir / "model.joblib").write_bytes(model_file.read())

# Import our prediction script.
inference_path = PROJECT_ROOT / "src" / "ticket_inference" / "inference.py"

spec = importlib.util.spec_from_file_location(
    "ticket_inference", inference_path
)
inference_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(inference_module)

serving_model = inference_module.model_fn(str(local_model_dir))

# Use one ticket from each production language.
sample_records = (
    production_df.groupby("language", sort=False)
    .head(1)[["ticket_id", "subject", "body", "language"]]
    .fillna("")
    .to_dict("records")
)

sample_request = "\n".join(
    json.dumps(record, ensure_ascii=False, allow_nan=False)
    for record in sample_records
)

parsed_records = inference_module.input_fn(
    sample_request, "application/jsonlines"
)
sample_predictions = inference_module.predict_fn(
    parsed_records, serving_model
)
response_body, response_type = inference_module.output_fn(
    sample_predictions, "application/jsonlines"
)

returned_records = [
    json.loads(line) for line in response_body.splitlines()
]

assert response_type == "application/jsonlines"
assert [r["ticket_id"] for r in returned_records] == [
    r["ticket_id"] for r in sample_records
]
assert all(
    r["predicted_queue"] in serving_model["pipeline"].classes_
    and 0.0 <= r["max_probability"] <= 1.0
    for r in returned_records
)

display(
    pd.DataFrame(returned_records)[
        ["language", "predicted_queue", "max_probability"]
    ].round(4)
)

,language,predicted_queue,max_probability
0,de,Technical Support,0.3737
1,en,Sales and Pre-Sales,0.2569
2,fr,IT Support,0.2941
3,pt,IT Support,0.3857
4,es,Product Support,0.2375


In [4]:
import sagemaker
from sagemaker.sklearn.model import SKLearnModel

INFERENCE_SOURCE_DIR = PROJECT_ROOT / "src" / "ticket_inference"

sagemaker_session = sagemaker.Session(
    boto_session=aws_session,
    default_bucket=PROJECT_BUCKET
)

# Verify the model URI still points to the version we checked.
model_location = urlparse(project_config["final_model"]["model_uri"])
model_head = s3_client.head_object(
    Bucket=model_location.netloc,
    Key=model_location.path.lstrip("/")
)
assert model_head["VersionId"] == (
    project_config["final_model"]["model_version_id"]
)

# Identify this combination of model and prediction code.
inference_source_hashes = {
    filename: hashlib.sha256(
        (INFERENCE_SOURCE_DIR / filename).read_bytes()
    ).hexdigest()
    for filename in ["inference.py", "requirements.txt"]
}

deployment_id = hashlib.sha256(
    (
        project_config["final_model"]["model_archive_sha256"]
        + json.dumps(inference_source_hashes, sort_keys=True)
    ).encode("utf-8")
).hexdigest()[:12]

batch_model_name = f"aai540-group3-routing-{deployment_id}"
batch_output_uri = (
    f"s3://{PROJECT_BUCKET}/batch/{deployment_id}/predictions/"
)

batch_model = SKLearnModel(
    name=batch_model_name,
    model_data=project_config["final_model"]["model_uri"],
    role=project_config["feature_store"]["execution_role_arn"],
    entry_point="inference.py",
    source_dir=str(INFERENCE_SOURCE_DIR),
    framework_version="1.4-2",
    py_version="py3",
    image_uri=project_config["managed_training"]["image_uri"],
    model_server_workers=1,
    sagemaker_session=sagemaker_session
)

batch_transformer = batch_model.transformer(
    instance_count=1,
    instance_type="ml.m5.xlarge",
    strategy="MultiRecord",
    assemble_with="Line",
    accept="application/jsonlines",
    max_payload=1,
    max_concurrent_transforms=1,
    output_path=batch_output_uri
)

project_config["deployment"].update({
    "model_name": batch_model_name,
    "deployment_id": deployment_id,
    "output_uri": batch_output_uri,
    "inference_source_sha256": inference_source_hashes,
    "instance_type": "ml.m5.xlarge"
})

config_path.write_text(
    json.dumps(project_config, indent=2),
    encoding="utf-8"
)

print("SageMaker model:", batch_model_name)
print("Prediction output:", batch_output_uri)

/opt/conda/lib/python3.12/site-packages/sagemaker/model.py:347: SageMakerV2DeprecationWarning: SKLearnModel is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
SKLearnModel is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
/opt/conda/lib/python3.12/site-packages/sagemaker/transformer.py:115: SageMakerV2DeprecationWarning: Transformer is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.shapes.TransformJob`.
See https://g

SageMaker model: aai540-group3-routing-ce725539a788
Prediction output: s3://aai540-group3-tickets-390568313988-us-east-1/batch/ce725539a788/predictions/


In [5]:
from datetime import datetime, timezone
from sagemaker.inputs import BatchDataCaptureConfig

deployment = project_config["deployment"]

# Prevent accidentally submitting the job twice.
if deployment.get("job_name"):
    raise RuntimeError(
        "A batch job is already recorded. Check its status "
        "before submitting another job."
    )

# Confirm the input file still matches the recorded S3 version.
input_location = urlparse(deployment["input"]["uri"])
input_head = s3_client.head_object(
    Bucket=input_location.netloc,
    Key=input_location.path.lstrip("/")
)
assert input_head["VersionId"] == deployment["input"]["version_id"]

batch_job_name = (
    "aai540-group3-batch-"
    + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
)

capture_uri = (
    f"s3://{PROJECT_BUCKET}/batch/"
    f"{deployment['deployment_id']}/capture/"
)

# Save the job name before starting predictions.
deployment.update({
    "job_name": batch_job_name,
    "capture_uri": capture_uri
})

config_path.write_text(
    json.dumps(project_config, indent=2),
    encoding="utf-8"
)

batch_transformer.transform(
    data=deployment["input"]["uri"],
    data_type="S3Prefix",
    content_type="application/jsonlines",
    split_type="Line",
    job_name=batch_job_name,
    batch_data_capture_config=BatchDataCaptureConfig(
        destination_s3_uri=capture_uri,
        generate_inference_id=False
    ),
    model_client_config={
        "InvocationsTimeoutInSeconds": 120,
        "InvocationsMaxRetries": 2
    },
    wait=False,
    logs=False
)

batch_details = sagemaker_client.describe_transform_job(
    TransformJobName=batch_job_name
)

print("Batch job:", batch_job_name)
print("Status:", batch_details["TransformJobStatus"])
print("Prediction output:", deployment["output_uri"])

INFO:sagemaker:Creating transform job with name: aai540-group3-batch-20261004-045734


Batch job: aai540-group3-batch-20261004-045734
Status: InProgress
Prediction output: s3://aai540-group3-tickets-390568313988-us-east-1/batch/ce725539a788/predictions/


In [7]:
batch_job_name = project_config["deployment"]["job_name"]

batch_details = sagemaker_client.describe_transform_job(
    TransformJobName=batch_job_name
)

status = batch_details["TransformJobStatus"]

print("Batch job:", batch_job_name)
print("Current status:", status)

if status == "Completed":
    print(
        "Predictions saved:",
        batch_details["TransformOutput"]["S3OutputPath"]
    )
elif status == "Failed":
    print("Failure reason:", batch_details.get("FailureReason"))
else:
    print("The job has not finished yet.")

Batch job: aai540-group3-batch-20261004-045734
Current status: Completed
Predictions saved: s3://aai540-group3-tickets-390568313988-us-east-1/batch/ce725539a788/predictions/


In [8]:
output_location = urlparse(
    batch_details["TransformOutput"]["S3OutputPath"]
)

output_keys = []
paginator = s3_client.get_paginator("list_objects_v2")

for page in paginator.paginate(
    Bucket=output_location.netloc,
    Prefix=output_location.path.lstrip("/")
):
    output_keys.extend(
        item["Key"]
        for item in page.get("Contents", [])
        if item["Key"].endswith(".out")
    )

if not output_keys:
    raise RuntimeError("No batch prediction output files were found.")

prediction_records = []
prediction_artifacts = []

for key in sorted(output_keys):
    response = s3_client.get_object(
        Bucket=output_location.netloc,
        Key=key
    )
    try:
        content = response["Body"].read()
    finally:
        response["Body"].close()

    prediction_records.extend(
        json.loads(line)
        for line in content.decode("utf-8").splitlines()
        if line.strip()
    )

    prediction_artifacts.append({
        "uri": f"s3://{output_location.netloc}/{key}",
        "version_id": response.get("VersionId"),
        "sha256": hashlib.sha256(content).hexdigest()
    })

production_predictions = pd.DataFrame(prediction_records)

# Verify coverage, valid predictions, and model identity.
assert len(production_predictions) == len(production_df)
assert production_predictions["ticket_id"].is_unique
assert set(production_predictions["ticket_id"]) == set(
    production_df["ticket_id"]
)
assert production_predictions["predicted_queue"].isin(
    serving_model["pipeline"].classes_
).all()
assert production_predictions["max_probability"].between(0, 1).all()
assert production_predictions["model_sha256"].eq(
    serving_model["model_sha256"]
).all()

prediction_dir = PROJECT_ROOT / "data" / "predictions"
prediction_dir.mkdir(parents=True, exist_ok=True)

production_predictions.to_csv(
    prediction_dir / "production_predictions.csv",
    index=False
)

project_config["deployment"].update({
    "status": batch_details["TransformJobStatus"],
    "prediction_count": len(production_predictions),
    "prediction_artifacts": prediction_artifacts
})

config_path.write_text(
    json.dumps(project_config, indent=2),
    encoding="utf-8"
)

print("Predictions verified:", len(production_predictions))
print("\nPredicted tickets per queue:")
print(production_predictions["predicted_queue"].value_counts())

Predictions verified: 17712

Predicted tickets per queue:
predicted_queue
Technical Support                  5039
Product Support                    3258
Customer Service                   3088
IT Support                         2269
Billing and Payments               1668
Returns and Exchanges               828
Service Outages and Maintenance     668
Sales and Pre-Sales                 586
Human Resources                     178
General Inquiry                     130
Name: count, dtype: int64


In [9]:
from sklearn.metrics import accuracy_score, f1_score

# Load the reserved ground-truth labels only after prediction is complete.
labels_info = project_config["s3_data"]["production_labels"]
labels_location = urlparse(labels_info["uri"])

response = s3_client.get_object(
    Bucket=labels_location.netloc,
    Key=labels_location.path.lstrip("/"),
    VersionId=labels_info["version_id"]
)
try:
    production_labels = pd.read_csv(
        response["Body"],
        dtype={"ticket_id": "string"}
    )
finally:
    response["Body"].close()

assert production_labels["ticket_id"].is_unique
assert production_labels["queue"].notna().all()
assert set(production_labels["ticket_id"]) == set(
    production_predictions["ticket_id"]
)

# Match labels and predictions by ticket ID, not row position.
production_evaluation = production_predictions.merge(
    production_labels[["ticket_id", "queue"]],
    on="ticket_id",
    how="inner",
    validate="one_to_one"
)

queue_labels = serving_model["pipeline"].classes_.tolist()
assert production_evaluation["queue"].isin(queue_labels).all()

production_metrics = {
    "evaluation_split": "simulated_production",
    "tickets": len(production_evaluation),
    "accuracy": float(accuracy_score(
        production_evaluation["queue"],
        production_evaluation["predicted_queue"]
    )),
    "macro_f1": float(f1_score(
        production_evaluation["queue"],
        production_evaluation["predicted_queue"],
        labels=queue_labels,
        average="macro",
        zero_division=0
    )),
    "weighted_f1": float(f1_score(
        production_evaluation["queue"],
        production_evaluation["predicted_queue"],
        labels=queue_labels,
        average="weighted",
        zero_division=0
    ))
}

monitoring_report_dir = PROJECT_ROOT / "reports" / "monitoring"
monitoring_report_dir.mkdir(parents=True, exist_ok=True)

production_report = {
    "batch_job_name": project_config["deployment"]["job_name"],
    "ground_truth_source": labels_info,
    "metrics": production_metrics
}

(monitoring_report_dir / "production_quality.json").write_text(
    json.dumps(production_report, indent=2),
    encoding="utf-8"
)

production_evaluation.to_csv(
    prediction_dir / "production_evaluation.csv",
    index=False
)

print("Production labels matched:", len(production_evaluation))
display(pd.DataFrame([
    project_config["final_model"]["test_metrics"],
    production_metrics
])[["evaluation_split", "accuracy", "macro_f1", "weighted_f1"]].round(4))

print("Production quality report saved.")

Production labels matched: 17712


,evaluation_split,accuracy,macro_f1,weighted_f1
0,test,0.4634,0.4464,0.4653
1,simulated_production,0.4591,0.4380,0.4605


Production quality report saved.


In [10]:
# Save the production-quality report to versioned S3 storage.
quality_report_path = monitoring_report_dir / "production_quality.json"
quality_report_key = (
    f"monitoring/{project_config['deployment']['job_name']}/"
    "production_quality.json"
)

s3_client.upload_file(
    str(quality_report_path),
    PROJECT_BUCKET,
    quality_report_key
)

report_metadata = s3_client.head_object(
    Bucket=PROJECT_BUCKET,
    Key=quality_report_key
)
assert report_metadata.get("VersionId") not in (None, "null")

project_config.setdefault("monitoring", {}).update({
    "production_quality": {
        "uri": f"s3://{PROJECT_BUCKET}/{quality_report_key}",
        "version_id": report_metadata["VersionId"],
        "metrics": production_metrics
    }
})

# Save the notebook package versions.
ml_requirements = (
    PROJECT_ROOT / "src" / "ticket_training" / "requirements.txt"
).read_text(encoding="utf-8")

(PROJECT_ROOT / "requirements-notebooks.txt").write_text(
    "sagemaker==2.257.6\n"
    "sagemaker-core==1.0.78\n"
    + ml_requirements,
    encoding="utf-8"
)

config_path.write_text(
    json.dumps(project_config, indent=2),
    encoding="utf-8"
)

print("Production quality report saved to S3.")
print("Notebook dependencies saved in requirements-notebooks.txt.")

Production quality report saved to S3.
Notebook dependencies saved in requirements-notebooks.txt.
